# NB_01_Load_API_Metadata
Reads the FHIR server's `/metadata` (CapabilityStatement) and fills `ctl_resource_config` and `ctl_watermark` in `WH_Metadata`.

- Resource list, endpoints, search parameters and interactions come **from the API**.
- Only your own choices (which resources, load order, critical flag, partition column) are in the settings cell.
- Safe to re-run: existing resources are updated in place (their `config_id`, `load_type` and watermark are kept); new ones are inserted with a 1900-01-01 watermark.

**Before running:** paste the warehouse SQL connection string into `sql_endpoint` (WH_Metadata → `...` → *Copy SQL connection string*).

In [ ]:
# PARAMETERS
source_id      = 1
warehouse_name = "WH_Metadata"
sql_endpoint   = ""      # e.g. abcd1234.datawarehouse.fabric.microsoft.com


In [ ]:
import json, struct, logging, requests, pyodbc
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("nb01")

assert sql_endpoint.strip(), "Set sql_endpoint in the parameters cell first."
sql_endpoint = sql_endpoint.strip().split(",")[0]

def get_conn():
    token = None
    for audience in ("pbi", "https://database.windows.net/"):
        try:
            token = notebookutils.credentials.getToken(audience)
            break
        except Exception as e:
            log.warning("Token audience %s failed: %s", audience, e)
    if not token:
        raise RuntimeError("Could not get an access token for the warehouse.")
    tb = token.encode("UTF-16-LE")
    tok_struct = struct.pack(f"<I{len(tb)}s", len(tb), tb)
    cs = (f"Driver={{ODBC Driver 18 for SQL Server}};Server={sql_endpoint},1433;"
          f"Database={warehouse_name};Encrypt=yes;TrustServerCertificate=no")
    return pyodbc.connect(cs, attrs_before={1256: tok_struct}, autocommit=True)

conn = get_conn()
cur = conn.cursor()
cur.execute("SELECT 1")
log.info("Connected to %s", warehouse_name)

In [ ]:
# YOUR CHOICES: only things the API cannot decide for you.
# kind: "dim" or "fact" (drives gold_proc name). partition: None = no partitioning.
SELECTED = {
    "Organization": {"order": 1, "critical": 1, "kind": "dim",  "partition": None},
    "Practitioner": {"order": 2, "critical": 1, "kind": "dim",  "partition": None},
    "Patient":      {"order": 3, "critical": 1, "kind": "dim",  "partition": None},
    "Encounter":    {"order": 4, "critical": 0, "kind": "fact", "partition": "lastUpdated_year_month"},
    "Observation":  {"order": 5, "critical": 0, "kind": "fact", "partition": "lastUpdated_year_month"},
    "Condition":    {"order": 6, "critical": 0, "kind": "fact", "partition": "lastUpdated_year_month"},
}
# FHIR standards applied by rule to every resource
PRIMARY_KEY        = "id"
INCREMENTAL_COLUMN = "meta.lastUpdated"
INCREMENTAL_PARAM  = "_lastUpdated"

In [ ]:
# 1. Read the source row
cur.execute("SELECT base_url, page_size FROM dbo.ctl_api_source WHERE source_id = ? AND is_active = 1", source_id)
row = cur.fetchone()
if not row:
    raise RuntimeError(f"No active row in ctl_api_source for source_id={source_id}. Run 02_seed_source.sql first.")
base_url, page_size = row[0].rstrip("/"), row[1]
log.info("Source: %s (page size %s)", base_url, page_size)

# 2. Call /metadata
r = requests.get(f"{base_url}/metadata", headers={"Accept": "application/fhir+json"}, timeout=(10, 90))
r.raise_for_status()
cap = r.json()
assert cap.get("resourceType") == "CapabilityStatement", "Unexpected response from /metadata"

catalog = {}
for res in cap["rest"][0]["resource"]:
    catalog[res["type"]] = {
        "search_params": sorted(p["name"] for p in res.get("searchParam", [])),
        "interactions":  sorted(i["code"] for i in res.get("interaction", [])),
    }
log.info("API exposes %d resource types", len(catalog))

In [ ]:
# 3. Validate the selection against what the API actually supports
valid = {}
for name, opts in SELECTED.items():
    info = catalog.get(name)
    if not info:
        log.error("%s: NOT exposed by the API, skipped", name); continue
    if "search-type" not in info["interactions"] or "read" not in info["interactions"]:
        log.error("%s: needs search-type and read, has %s, skipped", name, info["interactions"]); continue
    if INCREMENTAL_PARAM not in info["search_params"]:
        log.warning("%s: search parameter %s not listed, incremental loads may not work", name, INCREMENTAL_PARAM)
    valid[name] = {**opts, **info}
    log.info("%s OK: %d search params, interactions %s", name, len(info["search_params"]), info["interactions"])
if not valid:
    raise RuntimeError("No valid resources to load.")

In [ ]:
# 4. Upsert ctl_resource_config (keeps config_id and load_type of existing rows)
cur.execute("SELECT ISNULL(MAX(config_id), 0) FROM dbo.ctl_resource_config")
next_id = cur.fetchone()[0] + 1
inserted = updated = 0

for name, o in sorted(valid.items(), key=lambda kv: kv[1]["order"]):
    silver = "silver_" + name.lower()
    gold   = f"usp_Load_{o['kind']}_{name.lower()}"
    cur.execute("SELECT config_id FROM dbo.ctl_resource_config WHERE source_id = ? AND resource_name = ?", source_id, name)
    ex = cur.fetchone()
    if ex:
        cur.execute("""UPDATE dbo.ctl_resource_config
                       SET endpoint=?, primary_key=?, incremental_column=?, incremental_param=?,
                           silver_table=?, partition_column=?, gold_proc=?, load_order=?, is_critical=?, is_active=1
                       WHERE config_id=?""",
                    name, PRIMARY_KEY, INCREMENTAL_COLUMN, INCREMENTAL_PARAM,
                    silver, o["partition"], gold, o["order"], o["critical"], ex[0])
        updated += 1
    else:
        cur.execute("""INSERT INTO dbo.ctl_resource_config
                       (config_id, source_id, resource_name, endpoint, primary_key, incremental_column, incremental_param,
                        load_type, silver_table, partition_column, gold_proc, load_order, is_critical, is_active)
                       VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,1)""",
                    next_id, source_id, name, name, PRIMARY_KEY, INCREMENTAL_COLUMN, INCREMENTAL_PARAM,
                    "FULL", silver, o["partition"], gold, o["order"], o["critical"])
        next_id += 1
        inserted += 1
log.info("ctl_resource_config: %d inserted, %d updated", inserted, updated)

# 5. Seed watermarks only for configs that have none (first run = full load)
cur.execute("""INSERT INTO dbo.ctl_watermark (config_id, last_watermark, last_success_run_id, updated_at)
               SELECT c.config_id, CAST('1900-01-01' AS DATETIME2(6)), NULL, SYSUTCDATETIME()
               FROM dbo.ctl_resource_config c
               WHERE c.source_id = ?
                 AND NOT EXISTS (SELECT 1 FROM dbo.ctl_watermark w WHERE w.config_id = c.config_id)""", source_id)
log.info("Watermarks seeded for new resources")

In [ ]:
# 6. Verify
import pandas as pd
cfg = pd.read_sql("SELECT config_id, resource_name, endpoint, primary_key, incremental_param, load_type, silver_table, partition_column, gold_proc, load_order, is_critical, is_active FROM dbo.ctl_resource_config ORDER BY load_order", conn)
wm  = pd.read_sql("SELECT * FROM dbo.ctl_watermark ORDER BY config_id", conn)
display(cfg); display(wm)

sp = pd.DataFrame([{"resource": k, "interactions": ", ".join(v["interactions"]), "search_param_count": len(v["search_params"])}
                   for k, v in valid.items()])
display(sp)
assert len(cfg) >= len(valid) and len(wm) >= len(valid), "Row counts do not match the selection"
conn.close()